# Whisper Esperanto fine-tune — Colab (free T4)

Lab 7: fine-tune Whisper on a low-resource language (Esperanto) and compare against a CTC baseline (WER/CER).

**Why this notebook exists:** the original scripts loaded Common Voice from HuggingFace, but Mozilla removed Common Voice from HF in **October 2025**. The data now lives only on [Mozilla Data Collective (MDC)](https://mozilladatacollective.com) and is fetched with their API / Python SDK. The training scripts gained a `--local_data_dir` flag so they can read the extracted folder; this notebook wires it up on Colab.

**Free-tier plan:**
- Code (the `whisper_esperanto` folder) + trained models → **Google Drive** (small, persists).
- The Common Voice audio (large) → **Colab local disk** (`/content`, ~100 GB, wiped when the session ends). Re-download next session.
- Settings are sized for a **free T4**. On Colab Pro (L4/A100) raise `--batch_size` / `--max_train_samples` / `--max_steps`.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" evaluate jiwer accelerate peft librosa soundfile tensorboard

## 3. Get the training scripts

Upload your `whisper_esperanto` folder (the one containing `train_whisper.py`, `train_ctc.py` with the `--local_data_dir` edits) to your Google Drive first — e.g. to `MyDrive/whisper_esperanto`. Then run the cell (it mounts Drive and copies the folder to fast local disk).

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

REPO_DRIVE = '/content/drive/MyDrive/whisper_esperanto'   # <- where you uploaded the folder
REPO = '/content/whisper_esperanto'

assert os.path.isdir(REPO_DRIVE), f'Not found: {REPO_DRIVE} — upload the folder to Drive first.'
if os.path.isdir(REPO):
    shutil.rmtree(REPO)
shutil.copytree(REPO_DRIVE, REPO)
os.chdir(REPO)
print('Working in', os.getcwd())
print(sorted(os.listdir(REPO)))

## 4. Download the Common Voice Esperanto data from Mozilla Data Collective

1. Log in at <https://mozilladatacollective.com>, open the **Common Voice scripted-speech 26.0 (Esperanto)** dataset, and click **Agree to the terms of use** — the download API returns **403** until you do this on the website.
2. Go to **Account → Credentials** and create an **API key** (revoke the one you screenshotted earlier). Paste it below as `MDC_API_KEY`.
3. `DATASET_ID` is prefilled. If you get a **404**, it's wrong — copy the id from the dataset page URL: `https://mozilladatacollective.com/datasets/<ID>`.

The cell calls the documented endpoint `POST /api/datasets/{id}/download`, then downloads the presigned URL to local disk (resumable; org limit 30 downloads/day).

In [ ]:
import os, json, subprocess, urllib.request
from urllib.error import HTTPError

MDC_API_KEY = ''                          # <- your NEW MDC API key (revoke the old, leaked one!)
DATASET_ID = 'cmqim2wa300tonq071u17lvn'   # scripted-speech 26.0 Esperanto (from the dataset page URL)
os.makedirs('/content/mdc', exist_ok=True)
assert MDC_API_KEY, 'Fill in MDC_API_KEY.'

# 1) Ask MDC for a presigned download URL.
#    You must have clicked "Agree to the terms of use" on the dataset's web page first.
req = urllib.request.Request(
    f'https://mozilladatacollective.com/api/datasets/{DATASET_ID}/download',
    method='POST',
    headers={'Authorization': f'Bearer {MDC_API_KEY}',
             'Content-Type': 'application/json'})
try:
    with urllib.request.urlopen(req) as r:
        info = json.load(r)
except HTTPError as e:
    body = e.read().decode(errors='ignore')
    raise SystemExit(
        f'{e.code} {e.reason}: {body}\n\n'
        '403 -> open the dataset on mozilladatacollective.com and click "Agree to terms of use", then rerun.\n'
        '404 -> the DATASET_ID is wrong: copy the id from the dataset page URL (.../datasets/<ID>).')

url = info['downloadUrl']
fname = info.get('filename', 'dataset.tar.gz')
print('file:', fname, '| size (MB):', round(info.get('sizeBytes', 0) / 1e6, 1))

# 2) Download to local disk (-c = resume if interrupted).
dest = f'/content/mdc/{fname}'
subprocess.run(['wget', '-c', '-O', dest, url], check=True)
print('saved', dest)

In [ ]:
# If the SDK left a .tar/.tar.gz archive, extract it. (No-op if already a folder.)
import os, glob, tarfile

archives = glob.glob('/content/mdc/**/*.tar*', recursive=True)
for a in archives:
    print('extracting', a)
    with tarfile.open(a) as t:
        t.extractall(os.path.dirname(a))

In [ ]:
# Locate the folder that holds clips/ and train.tsv (the ASR corpus root).
import glob, os

cands = [os.path.dirname(p) for p in glob.glob('/content/mdc/**/train.tsv', recursive=True)]
if not cands:
    print('No train.tsv found. Extracted tree (send me this so I can adapt the loader):')
    for root, dirs, files in os.walk('/content/mdc'):
        print(root, '->', files[:8])
    raise SystemExit('Different dataset layout — see listing above.')
DATA_DIR = cands[0]
print('DATA_DIR =', DATA_DIR)
print('tsv files:', sorted(f for f in os.listdir(DATA_DIR) if f.endswith('.tsv')))
assert os.path.isdir(os.path.join(DATA_DIR, 'clips')), 'No clips/ folder next to the TSVs.'

## 5. Fine-tune Whisper (small + LoRA)

Esperanto has no Whisper language token, so we fine-tune with a proxy token (`--language polish`). Sized for a free T4; on a bigger GPU raise the numbers. Outputs (checkpoints + final model) go to Drive so they survive a disconnect.

In [ ]:
OUT = '/content/drive/MyDrive/whisper_eo_out'
os.environ['DATA_DIR'] = DATA_DIR
os.environ['OUT'] = OUT

!python train_whisper.py \
    --local_data_dir "$DATA_DIR" \
    --model openai/whisper-small \
    --language polish \
    --use_lora \
    --output_dir "$OUT/whisper-eo-small-lora" \
    --max_train_samples 3000 \
    --max_eval_samples 400 \
    --max_steps 800 \
    --batch_size 8 \
    --grad_accum 2 \
    --eval_steps 400 \
    --num_proc 2

## 6. CTC baseline (wav2vec2 XLS-R)

Same data, same text normalization → WER/CER directly comparable to the Whisper run above. This is the lab's "compare against CTC baseline" task.

In [ ]:
!python train_ctc.py \
    --local_data_dir "$DATA_DIR" \
    --model facebook/wav2vec2-xls-r-300m \
    --output_dir "$OUT/wav2vec2-eo" \
    --max_train_samples 3000 \
    --max_eval_samples 400 \
    --max_steps 1200 \
    --batch_size 4 \
    --grad_accum 4 \
    --eval_steps 400 \
    --num_proc 2

## 7. Results & next experiments

Each run prints a final `=== FINAL === {'eval_wer': ..., 'eval_cer': ...}` line — that's what goes in your report. Models are saved under `MyDrive/whisper_eo_out/`.

Experiments the flags let you run (maps to the lab tasks):

| Change | Tests |
|---|---|
| `--model openai/whisper-base` vs `-small` | model size effect |
| drop `--use_lora` | full fine-tune vs LoRA (needs more VRAM → lower batch) |
| `--language italian` / `croatian` / `polish` | multilingual transfer (proxy choice) |
| `--max_train_samples 2000 / 3000 / 6000` | data-scaling curve |

**If you hit CUDA out-of-memory on the T4:** lower `--batch_size` (e.g. 4) and raise `--grad_accum` to keep the effective batch, or drop `--max_train_samples`.